# Group Pipeline — Integrated Preprocessing & EDA

This notebook **combines Members 1–6** into one logical flow for Progress Review I (group component).

**Order:** Missing data → Categorical encoding → Outlier/duplicate removal → Feature engineering → Scaling → Feature selection / PCA → save artefacts.

All process diagrams, comparison bar plots, boxplots, heatmaps, and PCA charts are generated here and under each member notebook.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample, draw_process_flow,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
PIPELINE_STEPS = [
    "1 Missing\ndata",
    "2 Categorical\nencoding",
    "3 Outlier /\nduplicate",
    "4 Feature\nengineering",
    "5 Scaling",
    "6 SelectKBest\n+ PCA",
]
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


## 0. End-to-end process diagram


In [ ]:
fig, _ = draw_process_flow(
    PIPELINE_STEPS,
    title="Group preprocessing pipeline — full process flow",
    highlight=None,
)
fig.savefig(VIZ / "group_process_flow.png", dpi=150, bbox_inches="tight")
plt.show()


## Step 1–3 — Inventory, audit, encode, clean


In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

# 1) Missing / corrupt handling
inv = inventory_table(RAW)
discovered = discover_images(RAW)
valid, rejected = audit_images(RAW, discovered)
print("Inventory:"); display(inv)
print(f"Valid={len(valid)} | Rejected={len(rejected)}")

# 2) Categorical encoding
label_map = {c: i for i, c in enumerate(CLASSES)}
regions = sorted(valid["region"].unique().tolist())
region_map = {r: i for i, r in enumerate(regions)}
valid = valid.copy()
valid["label_encoded"] = valid["label"].map(label_map).astype(int)
valid["region_encoded"] = valid["region"].map(region_map).astype(int)

# 3) Duplicates + IQR outliers
unique, n_dup = remove_exact_duplicates(valid)
mask = iqr_mask(unique["pixels"]) & iqr_mask(unique["mean_g"])
clean = unique.loc[mask].reset_index(drop=True)
print(f"Duplicates removed={n_dup} | IQR removed={(~mask).sum()} | Clean={len(clean)}")
print(clean["label"].value_counts())


## Cleaning funnel & class / region comparison charts


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Funnel of sample counts through early steps
funnel_labels = ["Discovered", "Valid", "Unique", "Clean"]
funnel_vals = [len(discovered), len(valid), len(unique), len(clean)]
axes[0, 0].bar(funnel_labels, funnel_vals, color=["#7f7f7f", "#1f77b4", "#ff7f0e", "#2ca02c"])
axes[0, 0].set_title("Cleaning funnel (bar comparison)")
axes[0, 0].set_ylabel("Images")
for i, v in enumerate(funnel_vals):
    axes[0, 0].text(i, v + 5, str(v), ha="center")

# Expected vs found
x = np.arange(len(inv))
w = 0.35
short = [f.replace("_Region_Basil_Plant_Healthy", "").replace("Basil_Plant_", "") for f in inv["folder"]]
axes[0, 1].bar(x - w / 2, inv["expected"], width=w, label="Expected", color="#7f7f7f")
axes[0, 1].bar(x + w / 2, inv["found"], width=w, label="Found", color="#1f77b4")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(short, rotation=25, ha="right")
axes[0, 1].set_title("Expected vs found by folder")
axes[0, 1].legend()

# Clean class counts
class_counts = clean["label"].value_counts().reindex(CLASSES).fillna(0)
axes[1, 0].bar(class_counts.index, class_counts.values, color=["#2ca02c", "#d62728"])
axes[1, 0].set_title("Clean class counts")
axes[1, 0].set_ylabel("Count")

# Class × region after cleaning
ct = pd.crosstab(clean["region"], clean["label"]).reindex(columns=CLASSES).fillna(0)
ct.plot(kind="bar", ax=axes[1, 1], color=["#2ca02c", "#d62728"], rot=20)
axes[1, 1].set_title("Clean class × region comparison")
axes[1, 1].set_ylabel("Count")

fig.suptitle("Group pipeline — inventory & cleaning comparisons", fontsize=14)
fig.tight_layout()
fig.savefig(VIZ / "group_cleaning_comparisons.png", dpi=150, bbox_inches="tight")
plt.show()


## Step 4–6 — Features, scaling, selection, PCA


In [ ]:
# Stratified sample keeps the group demo responsive; set max_per_class=None for full data
max_per_class = 150
sample = clean if max_per_class is None else stratified_sample(clean, max_per_class)

X, y, kept = extract_feature_matrix(RAW, sample)
X_scaled = StandardScaler().fit_transform(X)
y_enc = LabelEncoder().fit_transform(y)

selector = SelectKBest(f_classif, k=min(20, X_scaled.shape[1]))
X_sel = selector.fit_transform(X_scaled, y_enc)
pca = PCA(n_components=min(5, X_sel.shape[1]), random_state=SEED)
X_pca = pca.fit_transform(X_sel)

print("Raw features:", X.shape)
print("After SelectKBest:", X_sel.shape)
print("After PCA:", X_pca.shape)
print("PCA variance:", np.round(pca.explained_variance_ratio_, 4))


## Feature / scaling / PCA comparison visualizations


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.boxplot(data=unique, x="label", y="mean_g", hue="label", order=CLASSES, ax=axes[0, 0], palette=["#2ca02c", "#d62728"], legend=False)
axes[0, 0].set_title("Green intensity (outlier view)")

axes[0, 1].hist(X[:, 0], bins=25, alpha=0.55, label="raw")
axes[0, 1].hist(X_scaled[:, 0], bins=25, alpha=0.55, label="scaled")
axes[0, 1].set_title("Feature[0] before/after scaling")
axes[0, 1].legend()

# PCA variance bars
pcs = np.arange(1, len(pca.explained_variance_ratio_) + 1)
axes[1, 0].bar(pcs, pca.explained_variance_ratio_, color="#ff7f0e", label="Per-PC")
axes[1, 0].plot(pcs, np.cumsum(pca.explained_variance_ratio_), "o-", color="#d62728", label="Cumulative")
axes[1, 0].set_title("PCA explained variance (comparison)")
axes[1, 0].set_xlabel("PC")
axes[1, 0].legend(fontsize=8)

for cls, color in zip(CLASSES, ["#2ca02c", "#d62728"]):
    m = y == cls
    axes[1, 1].scatter(X_pca[m, 0], X_pca[m, 1], s=16, alpha=0.75, c=color, label=cls)
axes[1, 1].set_title("PCA scatter after selection")
axes[1, 1].legend()

fig.suptitle("Group pipeline — feature & PCA EDA", fontsize=14)
fig.tight_layout()
fig.savefig(VIZ / "group_feature_pca_comparisons.png", dpi=150, bbox_inches="tight")
plt.show()

# Dimensionality reduction comparison
fig, ax = plt.subplots(figsize=(7, 4))
dims = ["Raw features", "SelectKBest", "PCA"]
dim_vals = [X.shape[1], X_sel.shape[1], X_pca.shape[1]]
ax.bar(dims, dim_vals, color=["#7f7f7f", "#1f77b4", "#2ca02c"])
ax.set_title("Dimensionality reduction funnel")
ax.set_ylabel("Feature dimensions")
for i, v in enumerate(dim_vals):
    ax.text(i, v + 0.5, str(v), ha="center")
fig.tight_layout()
fig.savefig(VIZ / "group_dimension_funnel.png", dpi=150, bbox_inches="tight")
plt.show()


## Save group artefacts + summary EDA panel


In [ ]:
# Tables / matrices
inv.to_csv(OUT / "group_folder_inventory.csv", index=False)
clean.to_csv(OUT / "group_cleaned_metadata.csv", index=False)
kept.assign(label_encoded=[label_map[v] for v in y]).to_csv(OUT / "group_feature_index.csv", index=False)
pd.DataFrame(X_scaled).to_csv(OUT / "group_features_scaled.csv", index=False)
pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]).assign(label=y).to_csv(
    OUT / "group_features_pca.csv", index=False
)
np.savez_compressed(OUT / "group_features.npz", X_scaled=X_scaled, X_pca=X_pca, y=y)

summary = {
    "discovered": int(len(discovered)),
    "valid": int(len(valid)),
    "rejected": int(len(rejected)),
    "duplicates_removed": int(n_dup),
    "clean": int(len(clean)),
    "feature_rows": int(X.shape[0]),
    "feature_dim": int(X.shape[1]),
    "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "label_map": label_map,
    "region_map": region_map,
}
(LOGS / "group_pipeline_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

# Combined EDA figure for viva walkthrough
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
class_counts = clean["label"].value_counts().reindex(CLASSES).fillna(0)
axes[0, 0].bar(class_counts.index, class_counts.values, color=["#2ca02c", "#d62728"])
axes[0, 0].set_title("1) Clean class counts")

sns.boxplot(data=unique, x="label", y="mean_g", hue="label", order=CLASSES, ax=axes[0, 1], palette=["#2ca02c", "#d62728"], legend=False)
axes[0, 1].set_title("2) Green intensity (outlier view)")

axes[1, 0].hist(X[:, 0], bins=25, alpha=0.55, label="raw")
axes[1, 0].hist(X_scaled[:, 0], bins=25, alpha=0.55, label="scaled")
axes[1, 0].set_title("3) Feature[0] before/after scaling")
axes[1, 0].legend()

for cls, color in zip(CLASSES, ["#2ca02c", "#d62728"]):
    m = y == cls
    axes[1, 1].scatter(X_pca[m, 0], X_pca[m, 1], s=16, alpha=0.75, c=color, label=cls)
axes[1, 1].set_title("4) PCA scatter after selection")
axes[1, 1].legend()

fig.suptitle("Group preprocessing pipeline — EDA summary", fontsize=14)
fig.tight_layout()
fig.savefig(VIZ / "group_pipeline_eda_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print("Group pipeline complete.")
print("Visualizations →", VIZ)
print("Outputs →", OUT)
print("Logs →", LOGS)


## Collaboration checklist
- [x] Member techniques integrated in dependency order  
- [x] Shared helpers in `src/preprocess_utils.py`  
- [x] Process-flow diagrams for each step  
- [x] Comparison bar plots, boxplots, heatmaps, PCA charts under `results/eda_visualizations/`  
- [x] Processed outputs under `results/outputs/`  
- [x] Run log under `results/logs/`  
